# Módulo 1: Introducción a la Inteligencia Artificial
## Notebook 3: Ejecutando nuestro primer LLM en Google Colab

Este notebook práctico te enseña a configurar un entorno de desarrollo en **Google Colab** para ejecutar LLMs de dos formas:
1. **Local (Open-Source):** Descargar y correr un modelo abierto utilizando la GPU de Google Colab.
2. **Basado en API (Nube):** Consumir el servicio de Google Gemini API.

### Objetivos de Aprendizaje
1. Habilitar la GPU T4 en Google Colab.
2. Ejecutar un LLM local (`Qwen2.5-1.5B-Instruct`) usando Hugging Face Transformers.
3. Conectar y realizar consultas seguras a la API de Google Gemini.
4. Implementar un bucle de conversación interactivo.

---  

## 1. Configuración del Entorno de Ejecución (GPU)

Para acelerar la generación de texto, utilizaremos una GPU gratuita de Colab.

### Habilitar GPU en Colab:
1. Ve a **Entorno de ejecución** -> **Cambiar tipo de entorno de ejecución**.
2. En **Acelerador de hardware**, selecciona **GPU T4**.
3. Guarda los cambios.

Ejecuta la siguiente celda para verificar la GPU:

In [1]:
# Verificar disponibilidad de GPU
!nvidia-smi

import torch
print("GPU Disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))


    

Mon Jun 22 22:56:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---  

## 2. Método 1: Correr un LLM Local (Código Abierto)

Usaremos el modelo `Qwen/Qwen2.5-1.5B-Instruct` por ser ligero, de acceso público (sin requerir aprobación) y eficiente en recursos.

### Instalar dependencias:

In [2]:
%pip install -q transformers accelerate

### Cargar Modelo y Tokenizador:

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Cargar tokenizador y modelo en FP16 para ahorrar memoria
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Crear pipeline de generación
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("¡Modelo cargado en GPU!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

¡Modelo cargado en GPU!


### Enviar un Prompt usando Chat Templates:

Damos estructura a la conversación definiendo roles (`system` y `user`).

In [4]:
messages = [
    {"role": "system", "content": "Eres un profesor divertido. Hablas español."},
    {"role": "user", "content": "¿Qué es una función en programación? Explícalo usando una analogía cotidiana."}
]

# Aplicar formato de chat
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Inferencia
outputs = generator(
    prompt,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

# Mostrar respuesta limpia
respuesta = outputs[0]["generated_text"][len(prompt):]
print(respuesta)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


¡Claro! Imagina que eres el dueño de una pequeña tienda de dulces. En tu tienda, tienes diferentes tipos de dulces como chocolates, galletas y fresas.

Una función en programación es similar a una caja mágica (como la tienda de dulces) que puede transformar los objetos o datos de entrada (como las frutas frescas) en otros objetos o datos (como pasteles concrecidos).

Por ejemplo:

1. **Función de chocolate**: Si le das al cliente 20 euros, esta función te regalará un pastel de chocolate.
   
2. **Función de galletas**: Si le das 30 euros, este también te regalará un pastel de galletas.

3. **Función de fresas**: Si le das 40 euros, esto podría ser una sorpresa para el cliente porque probablemente querrías algo más especial.

Cada vez que alguien pide una "función" específica, siempre obtienes un resultado diferente dependiendo del dinero que hayas invertido. Así que, en términos de programación, cada función tiene su propio código específico que se ejecuta cuando se llama,


---  

## 3. Método 2: Correr un LLM a través de API (Google Gemini API)

Usar APIs permite delegar el cómputo y ejecutar modelos más grandes sin necesidad de una GPU local.

### Configuración de la API Key:
1. En el panel lateral de Colab, ve al ícono de la llave (**Secrets**).
2. Crea un secreto llamado `GEMINI_API_KEY` con tu clave (consíguela gratis en [Google AI Studio](https://aistudio.google.com/)).
3. Activa el acceso al notebook para ese secreto.

### Instalar el SDK de Google:

In [ ]:
%pip install -q google-generativeai

### Configurar y Consultar Gemini:

In [ ]:
import google.generativeai as genai
from google.colab import userdata

try:
    # Obtener credencial segura
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)
    print("Gemini configurado con éxito.")
except Exception as e:
    print("Error al obtener credenciales:", e)

In [ ]:
model_gemini = genai.GenerativeModel('gemini-1.5-flash')

try:
    response = model_gemini.generate_content("Dame 3 ideas de proyectos cortos usando Python e IA.")
    print(response.text)
except Exception as e:
    print("Error en la consulta de API:", e)

---  

## 4. Comparativa: Local LLM vs API LLM

| Característica | Modelo Local (HF / Open-Source) | Modelo en la Nube (API) |
| :--- | :--- | :--- |
| **Privacidad** | Alta (datos locales). | Media (los datos viajan al servidor). |
| **Requisitos** | GPU con suficiente VRAM. | Conexión a internet constante. |
| **Control** | Modificación total y fine-tuning. | Limitado a los parámetros expuestos. |
| **Costo** | Cómputo propio (fijo). | Pago por uso (variables/tokens). |

---  

## 5. Ejercicios Prácticos

### Ejercicio 1: Experimentar con la Temperatura
Prueba con valores bajos (0.1) para respuestas predecibles y valores altos (1.2) para creatividad.

In [ ]:
prompt_ejercicio = "Escribe el inicio de un cuento sobre un robot que descubre cómo soñar."
messages_creative = [{"role": "user", "content": prompt_ejercicio}]
prompt_formatted = tokenizer.apply_chat_template(messages_creative, tokenize=False, add_generation_prompt=True)

# Generar con temperatura baja (0.1)
outputs_low = generator(prompt_formatted, max_new_tokens=100, do_sample=True, temperature=0.1, pad_token_id=tokenizer.eos_token_id)
print("Respuesta (Temp 0.1):", outputs_low[0]["generated_text"][len(prompt_formatted):])
print("\n" + "="*40 + "\n")

# Generar con temperatura alta (1.2)
outputs_high = generator(prompt_formatted, max_new_tokens=100, do_sample=True, temperature=1.2, pad_token_id=tokenizer.eos_token_id)
print("Respuesta (Temp 1.2):", outputs_high[0]["generated_text"][len(prompt_formatted):])

### Ejercicio 2: Rol de Sistema Personalizado
Modifica el contenido del rol `system` para personificar la respuesta (ej. un pirata informático).

In [ ]:
messages_custom = [
    {"role": "system", "content": "Escribe aquí tu rol personalizado (ej. Eres un pirata informático)."},
    {"role": "user", "content": "¿Cómo viaja la información por internet?"}
]
prompt_custom = tokenizer.apply_chat_template(messages_custom, tokenize=False, add_generation_prompt=True)

outputs_custom = generator(prompt_custom, max_new_tokens=150, do_sample=True, temperature=0.7, pad_token_id=tokenizer.eos_token_id)
print(outputs_custom[0]["generated_text"][len(prompt_custom):])

---  

## 6. Bucle de Interacción (Chat Interactivo)

Ejecuta la siguiente celda para chatear en tiempo real con el modelo local. Escribe **'salir'** para finalizar la sesión.

In [ ]:
# Inicializar historial de conversación
history = [
    {"role": "system", "content": "Eres un asistente de IA útil y conciso. Respondes en español."}
]

print("💬 Chat con el LLM iniciado. Escribe 'salir' para terminar.")

while True:
    user_input = input("\nTú: ")
    if user_input.strip().lower() in ["salir", "exit", "quit"]:
        print("Fin del chat.")
        break
    if not user_input.strip():
        continue
        
    # Agregar mensaje del usuario e inferir
    history.append({"role": "user", "content": user_input})
    prompt_chat = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
    
    outputs = generator(
        prompt_chat,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
    
    respuesta_chat = outputs[0]["generated_text"][len(prompt_chat):]
    print(f"AI: {respuesta_chat}")
    
    # Guardar respuesta en el historial
    history.append({"role": "assistant", "content": respuesta_chat})